**Purpose:** Merge technical + news + Reddit sentiment features into the portfolio dataset (dataset.parquet).

**Outputs:** `dataset.parquet`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT


In [170]:
import pandas as pd
import pickle
import talib
import os
import yfinance as yf
import numpy as np
import warnings
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)

non_tradable_days = str(PROJECT_ROOT / "01_data/aux/non_tradable_days.txt")

etfs_data = str(PROJECT_ROOT / "01_data/processed/etfs_0.pkl")
news_data = str(PROJECT_ROOT / "02_sentiment/news/as_feature/as_feature_data.parquet")
reddit_data = str(PROJECT_ROOT / "02_sentiment/reddit/as_feature/as_feature_data.parquet")

list_etfs = {
    'XLB': None,
    'XLC': None,
    'XLE': None,
    'XLF': None,
    'XLI': None,
    'XLK': None,
    'XLP': None,
    'XLRE': None,
    'XLU': None,
    'XLV': None,
    'XLY': None
}

In [171]:
df = pd.DataFrame(index=pd.date_range(start="2015-07-01", end="2025-06-30", freq="D"))

non_tradable_dates = pd.to_datetime(
    pd.read_csv(non_tradable_days, header=None)[0]
)

df = df.drop(non_tradable_dates, errors="ignore")

In [172]:
df

""
2015-07-01
2015-07-02
2015-07-06
2015-07-07
2015-07-08
...
2025-06-24
2025-06-25
2025-06-26
2025-06-27


# Inputs

### ETFs

| Sector                 | Ticker | ISIN                                         | Inception Date |
| ---------------------- | -----: | -------------------------------------------- | ---------------|
| Materials              |    XLB | **US81369Y1001**. ([SSGA][1], [Cbonds][2])   | 16/12/1998 |
| Communication Services |    XLC | **US81369Y8527**. ([SSGA][17], [Cbonds][18]) | 18/06/2018 |
| Energy                 |    XLE | **US81369Y5069**. ([SSGA][3], [Cbonds][4])   | 16/12/1998 |
| Financials             |    XLF | **US81369Y6059**. ([SSGA][5], [Cbonds][6])   | 16/12/1998 |
| Industrials            |    XLI | **US81369Y7040**. ([SSGA][7], [Cbonds][8])   | 16/12/1998 |
| Information Technology |    XLK | **US81369Y8030**. ([SSGA][9], [Cbonds][10])  | 16/12/1998 |
| Consumer Staples       |    XLP | **US81369Y3080**. ([SSGA][15], [Cbonds][16]) | 16/12/1998 |
| Real Estate            |   XLRE | **US81369Y8600**. ([SSGA][21], [Cbonds][22]) | 07/10/2015 |
| Utilities              |    XLU | **US81369Y8865**. ([SSGA][19], [Cbonds][20]) | 16/12/1998 |
| Health Care            |    XLV | **US81369Y2090**. ([SSGA][11], [Cbonds][12]) | 16/12/1998 |
| Consumer Discretionary |    XLY | **US81369Y4070**. ([SSGA][13], [Cbonds][14]) | 16/12/1998 |

[1]: https://www.ssga.com/nl/en_gb/intermediary/etfs/the-materials-select-sector-spdr-fund-xlb?utm_source=chatgpt.com "XLB: The Materials Select Sector SPDR® Fund"
[2]: https://cbonds.com/etf/273/?utm_source=chatgpt.com "XLB - Materials Select Sector SPDR Fund (USD), US81369Y1001"
[3]: https://www.ssga.com/us/en/intermediary/etfs/the-energy-select-sector-spdr-fund-xle?utm_source=chatgpt.com "XLE: The Energy Select Sector SPDR® Fund"
[4]: https://cbonds.com/etf/87/?utm_source=chatgpt.com "XLE - Energy Select Sector SPDR Fund (USD), US81369Y5069"
[5]: https://www.ssga.com/us/en/intermediary/etfs/the-financial-select-sector-spdr-fund-xlf?utm_source=chatgpt.com "XLF: The Financial Select Sector SPDR® Fund"
[6]: https://cbonds.com/etf/1/?utm_source=chatgpt.com "XLF - Financial Select Sector SPDR Fund (USD), US81369Y6059"
[7]: https://www.ssga.com/se/en_gb/intermediary/etfs/the-industrial-select-sector-spdr-fund-xli?utm_source=chatgpt.com "XLI: The Industrial Select Sector SPDR® Fund"
[8]: https://cbonds.com/etf/91/?utm_source=chatgpt.com "XLI - Industrial Select Sector SPDR Fund (USD), US81369Y7040"
[9]: https://www.ssga.com/us/en/intermediary/etfs/the-technology-select-sector-spdr-fund-xlk?utm_source=chatgpt.com "XLK: The Technology Select Sector SPDR® Fund"
[10]: https://cbonds.com/etf/21/?utm_source=chatgpt.com "XLK - Technology Select Sector SPDR Fund (USD), US81369Y8030"
[11]: https://www.ssga.com/us/en/intermediary/etfs/the-health-care-select-sector-spdr-fund-xlv?utm_source=chatgpt.com "XLV: The Health Care Select Sector SPDR® Fund"
[12]: https://cbonds.com/etf/89/?utm_source=chatgpt.com "XLV - Health Care Select Sector SPDR Fund (USD), US81369Y2090"
[13]: https://www.ssga.com/us/en/intermediary/etfs/the-consumer-discretionary-select-sector-spdr-fund-xly?utm_source=chatgpt.com "XLY: The Consumer Discretionary Select Sector SPDR® Fund"
[14]: https://cbonds.com/etf/79/?utm_source=chatgpt.com "XLY - Consumer Discretionary Select Sector SPDR Fund ... - Cbonds"
[15]: https://www.ssga.com/us/en/intermediary/etfs/the-consumer-staples-select-sector-spdr-fund-xlp?utm_source=chatgpt.com "XLP: The Consumer Staples Select Sector SPDR® Fund"
[16]: https://cbonds.com/etf/81/?utm_source=chatgpt.com "Consumer Staples Select Sector SPDR Fund (USD), US81369Y3080"
[17]: https://www.ssga.com/us/en/intermediary/etfs/the-communication-services-select-sector-spdr-fund-xlc?utm_source=chatgpt.com "XLC: The Communication Services Select Sector SPDR® Fund"
[18]: https://cbonds.com/etf/2271/?utm_source=chatgpt.com "XLC - The Communication Services Select Sector SPDR® Fund ..."
[19]: https://www.ssga.com/us/en/intermediary/etfs/the-utilities-select-sector-spdr-fund-xlu?utm_source=chatgpt.com "XLU: The Utilities Select Sector SPDR® Fund"
[20]: https://cbonds.com/etf/111/?utm_source=chatgpt.com "XLU - Utilities Select Sector SPDR Fund (USD), US81369Y8865"
[21]: https://www.ssga.com/us/en/intermediary/etfs/the-real-estate-select-sector-spdr-fund-xlre?utm_source=chatgpt.com "XLRE: The Real Estate Select Sector SPDR® Fund"
[22]: https://cbonds.com/etf/2273/?utm_source=chatgpt.com "The Real Estate Select Sector SPDR® Fund (USD), US81369Y8600"

In [173]:
with open(etfs_data, "rb") as f:
    etfs = pickle.load(f)

def compute_adjusted_ohlc(df: pd.DataFrame) -> pd.DataFrame:
    """
    Given a Yahoo Finance–style DataFrame with columns:
    Open, High, Low, Close, Adj Close, Dividends, Stock Splits, Capital Gains
    
    Returns a DataFrame with:
    Adj Open, Adj High, Adj Low, Adj Close
    
    Drops dividend-related columns.
    """

    df = df.copy()

    # Adjustment factor
    adj_factor = df["Adj Close"] / df["Close"]

    # Compute adjusted OHLC
    df["Adj Open"] = df["Open"] * adj_factor
    df["Adj High"] = df["High"] * adj_factor
    df["Adj Low"]  = df["Low"]  * adj_factor
    df["Adj Close"] = df["Adj Close"]  # explicit for clarity / checks

    # Keep only adjusted prices + volume (optional)
    cols_to_keep = ["Adj Open", "Adj High", "Adj Low", "Adj Close", "Volume"]
    df = df[cols_to_keep]

    return df

for key in list_etfs:
    etfs[key]["data"] = compute_adjusted_ohlc(etfs[key]["data"])

In [174]:
for key in list_etfs:
    df_prices = etfs[key]["data"]
    df_prices.index = df_prices.index.tz_localize(None)

    df_prices = df_prices.rename(
        columns=lambda c: f"{key}_{c.replace('Adj ', '').replace(' ', '_')}"
    )

    df = df.join(df_prices, how="left")

In [175]:
for key in list_etfs:

    df[f"{key}_SMA5"] = df[f"{key}_Close"].rolling(window=5).mean()
    df[f"{key}_SMA22"] = df[f"{key}_Close"].rolling(window=22).mean()
    df[f"{key}_SMA63"] = df[f"{key}_Close"].rolling(window=63).mean()
    df[f"{key}_SMA126"] = df[f"{key}_Close"].rolling(window=126).mean()
    df[f"{key}_SMA252"] = df[f"{key}_Close"].rolling(window=252).mean()

    df[f"{key}_MACD"] = df[f"{key}_Close"].ewm(span=12, adjust=False).mean() - df[f"{key}_Close"].ewm(span=26, adjust=False).mean()

    df[f"{key}_ADX"] = talib.ADX(df[f"{key}_High"], df[f"{key}_Low"], df[f"{key}_Close"], timeperiod=14)

    df[f"{key}_RSI"] = talib.RSI(df[f"{key}_Close"], timeperiod=14)

    df[f"{key}_CCI"] = talib.CCI(df[f"{key}_High"], df[f"{key}_Low"], df[f"{key}_Close"], timeperiod=14)

    df[f"{key}_OBV"] = talib.OBV(df[f"{key}_Close"], df[f"{key}_Volume"])

    df[f"{key}_ADI"] = talib.ADOSC(df[f"{key}_High"], df[f"{key}_Low"], df[f"{key}_Close"], df[f"{key}_Volume"])

    df[f"{key}_VolumeVariation"] = df[f"{key}_Volume"].pct_change()

    upper, middle, lower = talib.BBANDS(df[f"{key}_Close"], timeperiod=20, nbdevup=2, nbdevdn=2, matype=0)
    df[f"{key}_BBUpper"] = upper
    df[f"{key}_BBLower"] = lower

    df[f"{key}_5dayVolatility"] = df[f"{key}_Close"].rolling(window=5).std()
    df[f"{key}_22dayVolatility"] = df[f"{key}_Close"].rolling(window=22).std()
    df[f"{key}_63dayVolatility"] = df[f"{key}_Close"].rolling(window=63).std()

    df[f"{key}_RatioVolatility"] = df[f"{key}_63dayVolatility"] / df[f"{key}_22dayVolatility"]


In [176]:
vix = yf.download("^VIX", start="2015-07-01", end="2025-07-01")["Close"]

if vix.index[0].strftime("%Y-%m-%d") == "2015-07-01" and vix.index[-1].strftime("%Y-%m-%d") == "2025-06-30":
    df["VIXIndexClose"] = vix.values

else:
    raise ValueError("VIX data does not cover the expected date range.")

[*********************100%***********************]  1 of 1 completed


In [177]:
columns = {
    col.split("_", 1)[1] if any(col.startswith(f"{key}_") for key in list_etfs) else col
    for col in df.columns
}

{
    "Price": {
        "description": "Adjusted price level reflecting dividends and splits",
        "indicators": [
            "Close",
        ]
    },

    "OHLC": {
        "description": "Adjusted Open, High, Low, Close, and Volume",
        "indicators": [
            "Open",
            "High",
            "Low",
            "Close",
        ]
    },

    "Trend": {
        "description": "Direction, persistence, and strength of price movements",
        "indicators": [
            "SMA5",
            "SMA22",
            "SMA63",
            "SMA126",
            "SMA252",
            "MACD",
            "ADX"
        ]
    },

    "Momentum": {
        "description": "Short-term pressure, speed, and overbought/oversold dynamics",
        "indicators": [
            "RSI",
            "MACD",
            "CCI"
        ]
    },

    
    "Volume": {
        "description": "Market participation and accumulation/distribution dynamics",
        "indicators": [
            "Volume",
            "OBV",
            "ADI",
            "VolumeVariation"
        ]
    },

    "Volatility": {
        "description": "Price variability and risk",
        "indicators": [
            "BBUpper", "BBLower",
            "5dayVolatility",
            "22dayVolatility",
            "63dayVolatility",
            "RatioVolatility"
        ]
    },

    "Market": {
        "description": "Macro-level market context",
        "indicators": [
            "VIXIndexClose"
        ]
    },
}

''

''

In [178]:
df

,XLB_Open,XLB_High,XLB_Low,XLB_Close,XLB_Volume,XLC_Open,XLC_High,XLC_Low,XLC_Close,XLC_Volume,...,XLY_OBV,XLY_ADI,XLY_VolumeVariation,XLY_BBUpper,XLY_BBLower,XLY_5dayVolatility,XLY_22dayVolatility,XLY_63dayVolatility,XLY_RatioVolatility,VIXIndexClose
2015-07-01,39.886859,39.903183,39.609357,39.715462,5416300,NaN,NaN,NaN,NaN,NaN,...,6022600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.090000
2015-07-02,39.772602,39.862382,39.446130,39.593040,5138300,NaN,NaN,NaN,NaN,NaN,...,-236300.0,NaN,0.039236,NaN,NaN,NaN,NaN,NaN,NaN,16.790001
2015-07-06,39.266579,39.642022,39.046209,39.266579,6435300,NaN,NaN,NaN,NaN,NaN,...,-4286000.0,NaN,-0.352969,NaN,NaN,NaN,NaN,NaN,NaN,17.010000
2015-07-07,39.176791,39.266572,38.466713,39.135983,6859100,NaN,NaN,NaN,NaN,NaN,...,2675700.0,NaN,0.719066,NaN,NaN,NaN,NaN,NaN,NaN,16.090000
2015-07-08,38.809497,38.923762,38.246332,38.278980,7490800,NaN,NaN,NaN,NaN,NaN,...,-4149600.0,NaN,-0.019593,NaN,NaN,0.489454,NaN,NaN,NaN,19.660000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-24,87.129997,87.790001,87.010002,87.660004,4384100,104.580002,105.345001,104.400002,105.230003,4685500.0,...,568299000.0,-8.111411e+05,-0.310993,217.428872,209.022830,2.752882,2.159504,11.537932,5.342863,17.480000
2025-06-25,87.489998,87.489998,86.720001,86.820000,4372000,105.250000,105.800003,105.040001,105.209999,4401800.0,...,563648000.0,-1.416037e+06,0.075127,216.998808,209.116907,2.490886,2.149530,11.602164,5.397536,16.760000
2025-06-26,87.250000,88.059998,87.250000,87.779999,5254300,105.559998,106.480003,105.209999,106.430000,6375800.0,...,567419500.0,-7.377436e+05,-0.189099,217.076701,209.097603,2.133736,2.086504,11.696339,5.605712,16.590000
2025-06-27,87.809998,88.279999,87.400002,87.889999,4687800,106.510002,107.750000,106.489998,107.680000,6949700.0,...,571870500.0,1.004273e+06,0.180167,217.916915,208.664998,2.053099,2.283739,11.850897,5.189251,16.320000


### News

Classificação das notícias em 3 classes (positive, neutral, negative) para cada um dos 11 setores, usando o modelo ```experiments/sentiment_news/vf2 sectoremb```. As notícias vinham em intervalos de 15 minutos, e foram agrupadas por dia tendo em conta:

- passagem de UTC (GDEL) para EST (ETFs (NYSE)) e manter as notícias dentro do seu dia, a não ser que sejam antes da abertura do mercado (9:30 EST), caso em que são atribuídas ao dia anterior;

- dias em que mais de 20% dos intervalos de 15 minutos deram erro, foram discartados

Formato das colunas tem o formato ```{sector ETF}_news_sentiment_{sentiment}```

In [179]:
df_news = pd.read_parquet(news_data, engine="pyarrow")

df_news.index = pd.to_datetime(df_news.index)
df_news = df_news.sort_index()
df_news = df_news.replace({None: np.nan})

In [180]:
assets = list_etfs.keys()

news_transformed_data = {}

for i, row in df_news.iterrows():

    news_transformed_data[i] = {}

    for threshold in [0.00, 0.50, 0.75, 0.90]:

        for asset in assets:

            base_feat_name = f"{asset}_news_threshold{threshold:.2f}"

            pos = row[f"{asset}_news_sentiment_positive"]
            neg = row[f"{asset}_news_sentiment_negative"]
            is_missing = isinstance(pos, float) or isinstance(neg, float)

            if is_missing:
                news_transformed_data[i][f"_NEWS_NAN"] = 1

                news_transformed_data[i][f"{base_feat_name}_count_positive"] = 0
                news_transformed_data[i][f"{base_feat_name}_count_negative"] = 0
                news_transformed_data[i][f"{base_feat_name}_ratio_positive"] = 0.0
                news_transformed_data[i][f"{base_feat_name}_ratio_negative"] = 0.0
                news_transformed_data[i][f"{base_feat_name}_balance_ratio"] = 0.0
                news_transformed_data[i][f"{base_feat_name}_balance"] = 0.0
                
            else:
                # "LIVE" sentiment bucket 
                news_transformed_data[i][f"_NEWS_NAN"] = 0

                amount_pos = len([x for x in pos if x >= threshold])
                amount_neg = len([x for x in neg if x >= threshold])

                news_transformed_data[i][f"{base_feat_name}_count_positive"] = amount_pos
                news_transformed_data[i][f"{base_feat_name}_count_negative"] = amount_neg

                news_transformed_data[i][f"{base_feat_name}_ratio_positive"] = amount_pos / (amount_pos + amount_neg) if (amount_pos + amount_neg) > 0 else 0.0
                news_transformed_data[i][f"{base_feat_name}_ratio_negative"] = amount_neg / (amount_pos + amount_neg) if (amount_pos + amount_neg) > 0 else 0.0

                news_transformed_data[i][f"{base_feat_name}_balance"] = amount_pos - amount_neg

                news_transformed_data[i][f"{base_feat_name}_balance_ratio"] = (amount_pos - amount_neg) / (amount_pos + amount_neg) if (amount_pos + amount_neg) > 0 else 0.0

                
news_transformed_data = pd.DataFrame.from_dict(news_transformed_data, orient="index")


for asset in assets:
    for threshold in [0.00, 0.50, 0.75, 0.90]:
        base_feat_name = f"{asset}_news_threshold{threshold:.2f}"

        # Time-lagged features
        news_transformed_data[f"{base_feat_name}_balance_ratio_t-1"] = news_transformed_data[f"{base_feat_name}_balance_ratio"].shift(1)
        news_transformed_data[f"{base_feat_name}_balance_ratio_t-5"] = news_transformed_data[f"{base_feat_name}_balance_ratio"].shift(5)

        news_transformed_data[f"{base_feat_name}_balance_ratio_roll3"] = news_transformed_data[f"{base_feat_name}_balance_ratio"].rolling(window=3).mean()

        news_transformed_data[f"{base_feat_name}_balance_ratio_roll5"] = news_transformed_data[f"{base_feat_name}_balance_ratio"].rolling(window=5).mean()

        news_transformed_data[f"{base_feat_name}_balance_ratio_roll22"] = news_transformed_data[f"{base_feat_name}_balance_ratio"].rolling(window=22).mean()

        news_transformed_data[f"{base_feat_name}_balance_ratio_variation1"] = news_transformed_data[f"{base_feat_name}_balance_ratio"].diff(1)
        news_transformed_data[f"{base_feat_name}_balance_ratio_variation5"] = news_transformed_data[f"{base_feat_name}_balance_ratio"].diff(5)

In [181]:
df = df.join(news_transformed_data, how="left")

In [182]:
columns = {
    col.split("_", 1)[1] if any(col.startswith(f"{key}_") for key in list_etfs) else col
    for col in df.columns
}

{
    "_meta": {
        "format": "<sector>_news_threshold<threshold>_<feature>",
        "notes": "Extra column called '_NEWS_NAN' indicates whether the original sentiment data was missing (1) or present (0) for that day and asset.",
    },

    "NEWS_LIVE": {
        "description": "Real-time sentiment indicators based on news (GDELT) data, reflecting the current market sentiment for each asset",
        "indicators": [
            "balance",
            "balance_ratio",
            "count_positive", "count_negative",
            "ratio_positive", "ratio_negative"
        ]
    },

    "NEWS_HISTORICAL": {
        "description": "Historical sentiment indicators based on news (GDELT) data, reflecting the past market sentiment for each asset",
        "indicators": [
            "balance_ratio_roll22",
            "balance_ratio_roll3",
            "balance_ratio_roll5",
            "balance_ratio_t-1", "balance_ratio_t-5",
            "balance_ratio_variation1", "balance_ratio_variation5"
        ]
    },
}

''

''

In [183]:
df

,XLB_Open,XLB_High,XLB_Low,XLB_Close,XLB_Volume,XLC_Open,XLC_High,XLC_Low,XLC_Close,XLC_Volume,...,XLY_news_threshold0.75_balance_ratio_roll22,XLY_news_threshold0.75_balance_ratio_variation1,XLY_news_threshold0.75_balance_ratio_variation5,XLY_news_threshold0.90_balance_ratio_t-1,XLY_news_threshold0.90_balance_ratio_t-5,XLY_news_threshold0.90_balance_ratio_roll3,XLY_news_threshold0.90_balance_ratio_roll5,XLY_news_threshold0.90_balance_ratio_roll22,XLY_news_threshold0.90_balance_ratio_variation1,XLY_news_threshold0.90_balance_ratio_variation5
2015-07-01,39.886859,39.903183,39.609357,39.715462,5416300,NaN,NaN,NaN,NaN,NaN,...,NaN,-0.087209,-0.075305,0.411765,0.400000,0.003922,0.135686,NaN,-0.211765,-0.200000
2015-07-02,39.772602,39.862382,39.446130,39.593040,5138300,NaN,NaN,NaN,NaN,NaN,...,NaN,-0.162791,-0.750000,0.200000,1.000000,0.315033,0.002353,NaN,0.133333,-0.666667
2015-07-06,39.266579,39.642022,39.046209,39.266579,6435300,NaN,NaN,NaN,NaN,NaN,...,NaN,-0.460606,0.103876,1.000000,0.200000,0.300000,0.180000,NaN,-0.500000,0.300000
2015-07-07,39.176791,39.266572,38.466713,39.135983,6859100,NaN,NaN,NaN,NaN,NaN,...,NaN,0.348718,0.615385,0.500000,0.333333,0.833333,0.313333,NaN,0.500000,0.666667
2015-07-08,38.809497,38.923762,38.246332,38.278980,7490800,NaN,NaN,NaN,NaN,NaN,...,NaN,-0.463869,0.117032,1.000000,-0.333333,0.642857,0.465714,NaN,-0.571429,0.761905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-24,87.129997,87.790001,87.010002,87.660004,4384100,104.580002,105.345001,104.400002,105.230003,4685500.0,...,-0.236790,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.320009,0.000000,0.000000
2025-06-25,87.489998,87.489998,86.720001,86.820000,4372000,105.250000,105.800003,105.040001,105.209999,4401800.0,...,-0.222436,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.301292,0.000000,0.000000
2025-06-26,87.250000,88.059998,87.250000,87.779999,5254300,105.559998,106.480003,105.209999,106.430000,6375800.0,...,-0.205690,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.255838,0.000000,0.000000
2025-06-27,87.809998,88.279999,87.400002,87.889999,4687800,106.510002,107.750000,106.489998,107.680000,6949700.0,...,-0.178417,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.233110,0.000000,0.000000


### Reddit

Classificação de posts do reddit em 3 classes (positive, neutral, negative) para cada um dos 11 setores, usando o modelo ```experiments/sentiment_reddit/v4 model``` (config ```h5a8a3d210b```). As notícias vinham com timestamps, e foram agrupadas por dia tendo em conta:

- passagem de UTC para EST (ETFs (NYSE)) e manter as notícias dentro do seu dia, a não ser que sejam antes da abertura do mercado (9:30 EST), caso em que são atribuídas ao dia anterior;

Formato das colunas tem o formato ```{sector ETF}_reddit_sentiment_{sentiment}```

In [184]:
df_reddit = pd.read_parquet(reddit_data, engine="pyarrow")

df_reddit.index = pd.to_datetime(df_reddit.index)
df_reddit = df_reddit.sort_index()

In [185]:
assets = list_etfs.keys()

reddit_transformed_data = {}

for i, row in df_reddit.iterrows():

    reddit_transformed_data[i] = {}

    for threshold in [0.00, 0.50, 0.75, 0.90]:

        for asset in assets:

            base_feat_name = f"{asset}_reddit_threshold{threshold:.2f}"

            pos = row[f"{asset}_reddit_sentiment_positive"]
            neg = row[f"{asset}_reddit_sentiment_negative"]
            is_missing = isinstance(pos, float) or isinstance(neg, float)

            if is_missing:
                reddit_transformed_data[i][f"_REDDIT_NAN"] = 1

                reddit_transformed_data[i][f"{base_feat_name}_count_positive"] = 0
                reddit_transformed_data[i][f"{base_feat_name}_count_negative"] = 0
                reddit_transformed_data[i][f"{base_feat_name}_ratio_positive"] = 0.0
                reddit_transformed_data[i][f"{base_feat_name}_ratio_negative"] = 0.0
                reddit_transformed_data[i][f"{base_feat_name}_balance_ratio"] = 0.0
                reddit_transformed_data[i][f"{base_feat_name}_balance"] = 0.0
                
            else:
                # "LIVE" sentiment bucket 
                reddit_transformed_data[i][f"_REDDIT_NAN"] = 0

                amount_pos = len([x for x in pos if x >= threshold])
                amount_neg = len([x for x in neg if x >= threshold])

                reddit_transformed_data[i][f"{base_feat_name}_count_positive"] = amount_pos
                reddit_transformed_data[i][f"{base_feat_name}_count_negative"] = amount_neg

                reddit_transformed_data[i][f"{base_feat_name}_ratio_positive"] = amount_pos / (amount_pos + amount_neg) if (amount_pos + amount_neg) > 0 else 0.0
                reddit_transformed_data[i][f"{base_feat_name}_ratio_negative"] = amount_neg / (amount_pos + amount_neg) if (amount_pos + amount_neg) > 0 else 0.0

                reddit_transformed_data[i][f"{base_feat_name}_balance"] = amount_pos - amount_neg

                reddit_transformed_data[i][f"{base_feat_name}_balance_ratio"] = (amount_pos - amount_neg) / (amount_pos + amount_neg) if (amount_pos + amount_neg) > 0 else 0.0

                
reddit_transformed_data = pd.DataFrame.from_dict(reddit_transformed_data, orient="index")


for asset in assets:
    for threshold in [0.00, 0.50, 0.75, 0.90]:
        base_feat_name = f"{asset}_reddit_threshold{threshold:.2f}"

        # Time-lagged features
        reddit_transformed_data[f"{base_feat_name}_balance_ratio_t-1"] = reddit_transformed_data[f"{base_feat_name}_balance_ratio"].shift(1)
        reddit_transformed_data[f"{base_feat_name}_balance_ratio_t-5"] = reddit_transformed_data[f"{base_feat_name}_balance_ratio"].shift(5)

        reddit_transformed_data[f"{base_feat_name}_balance_ratio_roll3"] = reddit_transformed_data[f"{base_feat_name}_balance_ratio"].rolling(window=3).mean()

        reddit_transformed_data[f"{base_feat_name}_balance_ratio_roll5"] = reddit_transformed_data[f"{base_feat_name}_balance_ratio"].rolling(window=5).mean()

        reddit_transformed_data[f"{base_feat_name}_balance_ratio_roll22"] = reddit_transformed_data[f"{base_feat_name}_balance_ratio"].rolling(window=22).mean()

        reddit_transformed_data[f"{base_feat_name}_balance_ratio_variation1"] = reddit_transformed_data[f"{base_feat_name}_balance_ratio"].diff(1)
        reddit_transformed_data[f"{base_feat_name}_balance_ratio_variation5"] = reddit_transformed_data[f"{base_feat_name}_balance_ratio"].diff(5)

In [186]:
df = df.join(reddit_transformed_data, how="left")

In [187]:
columns = {
    col.split("_", 1)[1] if any(col.startswith(f"{key}_") for key in list_etfs) else col
    for col in df.columns
}

{
    "_meta": {
        "format": "<sector>_reddit_threshold<threshold>_<feature>",
        "notes": "Extra column called '_REDDIT_NAN' indicates whether the original sentiment data was missing (1) or present (0) for that day and asset.",
    },

    "REDDIT_LIVE": {
        "description": "Real-time sentiment indicators based on Reddit posts, reflecting the current market sentiment for each asset",
        "indicators": [
            "balance",
            "balance_ratio",
            "count_positive", "count_negative",
            "ratio_positive", "ratio_negative"
        ]
    },

    "REDDIT_HISTORICAL": {
        "description": "Historical sentiment indicators based on Reddit posts, reflecting the past market sentiment for each asset",
        "indicators": [
            "balance_ratio_roll22",
            "balance_ratio_roll3",
            "balance_ratio_roll5",
            "balance_ratio_t-1", "balance_ratio_t-5",
            "balance_ratio_variation1", "balance_ratio_variation5"
        ]
    },
}

''

''

In [188]:
df

,XLB_Open,XLB_High,XLB_Low,XLB_Close,XLB_Volume,XLC_Open,XLC_High,XLC_Low,XLC_Close,XLC_Volume,...,XLY_reddit_threshold0.75_balance_ratio_roll22,XLY_reddit_threshold0.75_balance_ratio_variation1,XLY_reddit_threshold0.75_balance_ratio_variation5,XLY_reddit_threshold0.90_balance_ratio_t-1,XLY_reddit_threshold0.90_balance_ratio_t-5,XLY_reddit_threshold0.90_balance_ratio_roll3,XLY_reddit_threshold0.90_balance_ratio_roll5,XLY_reddit_threshold0.90_balance_ratio_roll22,XLY_reddit_threshold0.90_balance_ratio_variation1,XLY_reddit_threshold0.90_balance_ratio_variation5
2015-07-01,39.886859,39.903183,39.609357,39.715462,5416300,NaN,NaN,NaN,NaN,NaN,...,NaN,1.0,NaN,0.0,NaN,NaN,NaN,NaN,1.000000,NaN
2015-07-02,39.772602,39.862382,39.446130,39.593040,5138300,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,NaN,1.0,NaN,0.666667,NaN,NaN,0.000000,NaN
2015-07-06,39.266579,39.642022,39.046209,39.266579,6435300,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,-1.0,0.0,1.0,0.000000,0.200000,NaN,0.000000,-1.000000
2015-07-07,39.176791,39.266572,38.466713,39.135983,6859100,NaN,NaN,NaN,NaN,NaN,...,NaN,1.0,0.0,0.0,1.0,0.333333,0.200000,NaN,1.000000,0.000000
2015-07-08,38.809497,38.923762,38.246332,38.278980,7490800,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,1.0,1.0,0.0,0.666667,0.400000,NaN,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-24,87.129997,87.790001,87.010002,87.660004,4384100,104.580002,105.345001,104.400002,105.230003,4685500.0,...,0.297619,0.0,0.0,-0.2,0.0,0.266667,0.360000,0.203030,0.200000,0.000000
2025-06-25,87.489998,87.489998,86.720001,86.820000,4372000,105.250000,105.800003,105.040001,105.209999,4401800.0,...,0.206710,-1.0,-2.0,0.0,1.0,-0.400000,-0.040000,0.112121,-1.000000,-2.000000
2025-06-26,87.250000,88.059998,87.250000,87.779999,5254300,105.559998,106.480003,105.209999,106.430000,6375800.0,...,0.206710,1.0,0.0,-1.0,0.0,-0.333333,-0.040000,0.112121,1.000000,0.000000
2025-06-27,87.809998,88.279999,87.400002,87.889999,4687800,106.510002,107.750000,106.489998,107.680000,6949700.0,...,0.252165,0.0,-1.0,0.0,1.0,-0.444444,-0.306667,0.142424,-0.333333,-1.333333


# Save

In [ ]:
# df.to_parquet("dataset.parquet")

In [ ]:
# with open("dataset_features.txt", "w") as f:
#     f.write("\n".join(sorted(list(columns))))